In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import holidays
import hashlib
import tools

In [2]:
df_bez_nigdy_nie_sprzedane = pd.read_parquet("dane/interim/fact_inka_records_2023_2026_hard_bez_nigdy_nie_sprzedane.parquet")

### Towary wazone 

In [3]:
# ============================================================
# SOFT FILTER — Kryterium 2: Towary ważone (per TowId)
# Krok 1: % transakcji z ułamkową ilością per TowId
# ============================================================

wazone_check = df_bez_nigdy_nie_sprzedane.groupby('TowId').agg(
    LiczbaTransakcji=('IloscPlus', 'count'),
    LiczbaUlamkowych=('IloscPlus', lambda x: (x % 1 != 0).sum()),
).reset_index()

wazone_check['PctUlamkowych'] = (
    wazone_check['LiczbaUlamkowych'] / wazone_check['LiczbaTransakcji'] * 100
)

print(wazone_check['PctUlamkowych'].describe())

count    12481.000000
mean         1.310328
std         10.598002
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max        100.000000
Name: PctUlamkowych, dtype: float64


In [4]:


def czy_nazwa_sugeruje_wage(nazwa):
    nazwa = str(nazwa).upper().strip()
    
    # Wyklucz przypadki "liczba+KG" (stała gramatura opakowania, np. "0,5KG", "1,65KG")
    bez_gramatury = re.sub(r'\d+[\.,]?\d*\s*KG\b', '', nazwa)
    
    # Szukamy samodzielnego "KG" jako jednostki sprzedaży (bez liczby bezpośrednio przed nim)
    return bool(re.search(r'\bKG\b', bez_gramatury))

# Szybki test kontrolny
testy = [
    "Chleb mieszany 0.6 kg PRECELEK",         # False - gramatura
    "PROSZEK PERSIL UNIVERSAL 1,65KG HENKEL", # False - gramatura
    "MĄKA ZIEMNIACZANA 0,5KG",                # False - gramatura
    "SURÓWKA KG",                              # True - waga
    "BAKŁAŻAN KG POLSKA",                      # True - waga
    "SAŁATKA WIELKANOCNA KG GRZEŚKOWIAK",      # True - waga
]
for t in testy:
    print(f"{czy_nazwa_sugeruje_wage(t)}: {t}")

False: Chleb mieszany 0.6 kg PRECELEK
False: PROSZEK PERSIL UNIVERSAL 1,65KG HENKEL
False: MĄKA ZIEMNIACZANA 0,5KG
True: SURÓWKA KG
True: BAKŁAŻAN KG POLSKA
True: SAŁATKA WIELKANOCNA KG GRZEŚKOWIAK


In [5]:
nazwy_do_sprawdzenia = df_bez_nigdy_nie_sprzedane[['TowId', 'NazwaTow']].drop_duplicates(subset='TowId')
nazwy_do_sprawdzenia['NazwaSugerujeWage'] = nazwy_do_sprawdzenia['NazwaTow'].apply(czy_nazwa_sugeruje_wage)

wazone_check_pelne = wazone_check.merge(nazwy_do_sprawdzenia, on='TowId', how='left')

wazone_check_pelne['JestWazony'] = (
    (wazone_check_pelne['PctUlamkowych'] >= 50) |
    (wazone_check_pelne['NazwaSugerujeWage'])
)

lista_wazonych_towid = set(wazone_check_pelne[wazone_check_pelne['JestWazony']]['TowId'])
print(f"Ważonych TowId (finalne kryterium): {len(lista_wazonych_towid)}")

dodane_przez_nazwe = wazone_check_pelne[
    (wazone_check_pelne['NazwaSugerujeWage']) & 
    (wazone_check_pelne['PctUlamkowych'] < 50)
]
print(f"\nDodatkowo złapane przez samodzielne 'KG' (bez liczby): {len(dodane_przez_nazwe)}")
print(dodane_przez_nazwe[['TowId', 'NazwaTow', 'PctUlamkowych']].sort_values('NazwaTow'))

Ważonych TowId (finalne kryterium): 199

Dodatkowo złapane przez samodzielne 'KG' (bez liczby): 16
       TowId                                 NazwaTow  PctUlamkowych
5390   52141                       BAKŁAŻAN KG POLSKA      16.666667
4908   51075                     CUKIERKI PROMOCJA KG       0.000000
9041   67440  FILETY ŚLEDZIOWE Z CEBULA KG SUPER FISH       0.000000
9059   77451      KAPUSTA Z GRZYBAMI KG KUCHNIA POLKI      33.333333
10393  78987  KAWA ZIAR JACOBS BRAZIL&COLUMBIA KG JDE       0.000000
10394  78988       KAWA ZIAR JACOBS LAOS&INDIA KG JDE       0.000000
9039   67438             KRAJANKA PO ŻYDOWSKU KG SEKO      47.619048
8937   67313    KROKIETY KAPUSTA GRZYB KG GRZEŚKOWIAK      31.034483
8938   67314         KROKIETY Z MIĘSEM KG GRZEŚKOWIAK      15.384615
9331   77799                MAKARON PENNE KG SPAR NO1       0.000000
9332   77800            MAKARON SPAGHETTI KG SPAR NO1       0.000000
5958   53355                         MELON MIODOWY KG      49.315068
6755

In [6]:
df_bez_nigdy_nie_sprzedane['JestWazony'] = df_bez_nigdy_nie_sprzedane['TowId'].isin(lista_wazonych_towid)

print(df_bez_nigdy_nie_sprzedane['JestWazony'].value_counts())

nazwy_wazonych = (
    df_bez_nigdy_nie_sprzedane[df_bez_nigdy_nie_sprzedane['TowId'].isin(lista_wazonych_towid)]
    [['TowId', 'NazwaTow', 'NazwaAsort']]
    .drop_duplicates(subset='TowId')
)
print(nazwy_wazonych['NazwaAsort'].value_counts())

JestWazony
False    3768351
True      288981
Name: count, dtype: int64
NazwaAsort
CUKIERKI WAGA                    43
SERY WAGA /nabiał                34
WARZYWA                          32
OWOCE                            20
CUKIERKI                         16
WĘDLINY WAGA                     10
CIASTKA WAGA                      8
PIEROGI KOPYTKA KROKIETY INNE     6
WĘDLINY PACZKOWANE                4
RYBY MROŻONE WAGA                 3
RYBY WĘDZONE WAGA /ryby           3
WARZYWA KISZONE                   3
PIECZYWO                          2
PRZETWORY RYBNE /ryby             2
MARKA WŁASNA SPAR                 2
KAWY                              2
SURÓWKI I SAŁATKI                 2
MIĘSO DROBIOWE WAGA               1
WIELKANOCNE                       1
MIĘSO, WĘDLINY I GARMAŻERKA       1
***PRZECENY                       1
CHLEBY                            1
ŚWIĄTECZNE                        1
CIASTKA                           1
Name: count, dtype: int64


In [7]:
df_bez_nigdy_nie_sprzedane['JestWazony'] = df_bez_nigdy_nie_sprzedane['TowId'].isin(lista_wazonych_towid)

print(df_bez_nigdy_nie_sprzedane['JestWazony'].value_counts())

nazwy_wazonych = (
    df_bez_nigdy_nie_sprzedane[df_bez_nigdy_nie_sprzedane['TowId'].isin(lista_wazonych_towid)]
    [['TowId', 'NazwaTow', 'NazwaAsort']]
    .drop_duplicates(subset='TowId')
)
print(nazwy_wazonych['NazwaAsort'].value_counts())

JestWazony
False    3768351
True      288981
Name: count, dtype: int64
NazwaAsort
CUKIERKI WAGA                    43
SERY WAGA /nabiał                34
WARZYWA                          32
OWOCE                            20
CUKIERKI                         16
WĘDLINY WAGA                     10
CIASTKA WAGA                      8
PIEROGI KOPYTKA KROKIETY INNE     6
WĘDLINY PACZKOWANE                4
RYBY MROŻONE WAGA                 3
RYBY WĘDZONE WAGA /ryby           3
WARZYWA KISZONE                   3
PIECZYWO                          2
PRZETWORY RYBNE /ryby             2
MARKA WŁASNA SPAR                 2
KAWY                              2
SURÓWKI I SAŁATKI                 2
MIĘSO DROBIOWE WAGA               1
WIELKANOCNE                       1
MIĘSO, WĘDLINY I GARMAŻERKA       1
***PRZECENY                       1
CHLEBY                            1
ŚWIĄTECZNE                        1
CIASTKA                           1
Name: count, dtype: int64


In [8]:
df_bez_nigdy_nie_sprzedane.to_parquet(
    "dane/interim/fact_inka_hard_flagged_wazone.parquet",
    compression='zstd',
    index=False
)
print(f"Zapisano: {df_bez_nigdy_nie_sprzedane.shape}")

Zapisano: (4057332, 33)


In [9]:
# suma kontrolna
nazwa_pliku = "fact_inka_hard_flagged_wazone.parquet"
moj_hash = tools.hash_danych_bezpieczny(f"dane/interim/{nazwa_pliku}")
print(f"Mój hash (posortowane):   {nazwa_pliku}   {moj_hash}")

Mój hash (posortowane):   fact_inka_hard_flagged_wazone.parquet   c1208aa8a593c5ba22da108f49167ae17b9e99664d9fb53d129ea13115025ca5
